In [21]:
import os.path as op
import os
import numpy as np
import glob

In [22]:
db_path = '/home/dnl/Dropbox_ASU/Decision_Neuroscience_Lab/'
home_dir =  '/home/dnl/habitization/'
subs = db_path + 'Habitization/subjects.txt'
subs = list(np.loadtxt(subs,str))

ignore_subs = ['HAB02', 'HAB03', 'HAB05', 'HAB08', 'HAB10', 'HAB11', 'HAB12', 'HAB13', 'HAB14', 'HAB16', 'HAB07', 'HAB18', 'HAB19', 'HAB20', 'HAB21', 'HAB22', 'HAB23', 'HAB24', 'HAB25']
#First analysis included subjects 'HAB02', 'HAB03', 'HAB05', 'HAB08', 'HAB10', 'HAB11', 'HAB12', 'HAB13', 'HAB14', 'HAB16'
#Second analysis included subjects 'HAB01', 'HAB07', 'HAB18', 'HAB19', 'HAB20', 'HAB21', 'HAB22', 'HAB23', 'HAB24', 'HAB25'
subs = [x for x in subs if x not in ignore_subs]

In [23]:
def make_new_dir(dir_name):
    if not op.exists(dir_name):
        os.mkdir(dir_name)

In [24]:
sesh_map = {'Session1':'a','Session5':'b'}
for sub in subs:
    if True:#sub == 'HAB10':
        
        #Make all the directories we need##
        sub_dir = home_dir + 'data/'+ sub
        make_new_dir(sub_dir)
        
        func_dir = sub_dir + '/func/'
        make_new_dir(func_dir)
            
        anat_dir = sub_dir + '/anat/'
        make_new_dir(anat_dir)
            
        cal_dir = sub_dir + '/cal/'
        make_new_dir(cal_dir)
       
        timing_files_dir = sub_dir + '/timing_files/'
        make_new_dir(timing_files_dir)
        
        onsets_dir = timing_files_dir + '/onsets/'
        make_new_dir(onsets_dir)
        
        fmap_dir = sub_dir + '/fmap/'
        make_new_dir(fmap_dir)

        
        #loop through sessions
        for sesh in sesh_map.keys():
            fmri_path = db_path + '/fMRI_Data/Habitization/' + sub + sesh +'/'

            ##First copy functional data##
            epis = glob.glob(fmri_path + '/EP*gz') #glob(pathname) returns list of paths that match pathname
            if len(epis) != 6:
                print sub,sesh,'epi'

            for epi in epis:
                run = epi.split('/')[-1].split('.')[0][-1]

                new_f = func_dir + 'run_' + sesh_map[sesh] + run + '.nii.gz'
                if op.exists(new_f):
                    os.remove(new_f)
                    
                cmd = ['ln','-s',epi, new_f] #creates a link
                os.system(' '.join(cmd))
            
            ##Next copy anatomical data##
            T1s = glob.glob(fmri_path + '/*BRAVO*')
            if len(T1s) < 1:
                print sub,sesh,'T1'
                
            for t1 in T1s:
                new_f = anat_dir + 'T1_' + sesh_map[sesh] + '.nii.gz'
                if op.exists(new_f):
                    os.remove(new_f)
                
                cmd = ['ln','-s',t1, new_f] #creates a link
                os.system(' '.join(cmd))
            
            ##Next copy cal
            cals = glob.glob(fmri_path + '/*Field*')
            cals.sort()
            if len(cals) != 2:
                print sub,sesh,'cal'
                
            for n,cal in enumerate(cals):
                new_f = cal_dir + 'cal_' + sesh_map[sesh] + str(n) + '.nii.gz'
                if op.exists(new_f):
                    os.remove(new_f)
                
                cmd = ['ln','-s',cal, new_f]
                os.system(' '.join(cmd))  
            
            ##Next copy timing files
            onset_path = db_path + '/Habitization/timing_files/original/' + sub
            onsets = glob.glob(onset_path + '/*txt')
            if len(onsets) != 12:
                print sub,sesh,'onsets'
                
            for o in onsets:
                lowercase = {'B':'b','A':'a'}
                file_id = o.split('/')[-1].split('_')[1:3]
                run = file_id[0][-1]
                sesh = lowercase[file_id[1]]
                new_f = onsets_dir + 'run_' + sesh + run + '.txt'

                if op.exists(new_f):
                    os.remove(new_f)
                
                
                cmd = ['ln','-s',o, new_f]
                os.system(' '.join(cmd))
            